In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import glob
import kagglehub
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt


class SUIMDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None, target_transform=None):
        # Store directories
        self.images_dir = images_dir  # Folder containing .jpg images
        self.masks_dir = masks_dir  # Folder containing .png masks

        # Collect all image paths
        self.image_paths = glob.glob(os.path.join(self.images_dir, "*.jpg"))  # Get all .jpg files
        self.image_paths.sort()  # Sort for deterministic ordering

        # Build a mapping from "stem" -> mask_path to robustly pair image/mask
        mask_paths = glob.glob(os.path.join(self.masks_dir, "*.png"))  # Get all .png mask files
        mask_paths.sort()  # Sort for determinism

        self.mask_map = {}
        for mp in mask_paths:
            stem = os.path.splitext(os.path.basename(mp))[0]  # Remove extension
            self.mask_map[stem] = mp  # Save mapping

        # Filter image_paths to those that have a corresponding mask
        paired_image_paths = []  # Will store only images that have masks
        paired_mask_paths = []  # Will store corresponding masks in same order
        for ip in self.image_paths:
            stem = os.path.splitext(os.path.basename(ip))[0]  # Image stem
            if stem in self.mask_map:
                paired_image_paths.append(ip)  # Keep image
                paired_mask_paths.append(self.mask_map[stem])  # Keep its mask

        self.image_paths = paired_image_paths  # Replace with paired list
        self.mask_paths = paired_mask_paths  # Store aligned mask list

        # Store transforms
        self.transform = transform  # Transformations for images
        self.target_transform = target_transform  # Transformations for masks

    def __len__(self):
        return len(self.image_paths)  # Number of paired samples

    def __getitem__(self, idx):
        # Load image and mask paths
        image_path = self.image_paths[idx]  # Get image path
        mask_path = self.mask_paths[idx]  # Get mask path

        # Load image as RGB
        image = Image.open(image_path).convert("RGB")  # Ensure 3 channels

        # Load mask as single channel (L)
        mask = Image.open(mask_path).convert("L")  # Keep as grayscale label map

        # Apply image transforms (if provided)
        if self.transform:
            image = self.transform(image)  # Apply preprocessing/augmentation to image

        # Apply mask transforms (if provided)
        if self.target_transform:
            mask = self.target_transform(mask)  # Apply resizing with nearest, then tensor conversion

        # Ensure mask is integer labels and remap to consecutive IDs
        # mask_transforms below uses PILToTensor -> shape [1,H,W] with dtype uint8; convert to long before remap
        mask = remap_mask(mask)  # Remap unique values to consecutive range

        return image, mask  # Return image and mask



image_transforms = transforms.Compose([
    transforms.ToTensor(),  # Convert PIL image to tensor in [0,1]
    transforms.Resize((256, 256)),  # Resize to a fixed size for batching
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Preserve label IDs
    transforms.PILToTensor(),  # Convert to tensor (keeps integer values)
])



candidate_roots = [
    os.path.join(path, "dataset"),  # Common: dataset/
    path,  # Fallback: directly under path/
]

images_dir = None  # Will be set once found
masks_dir = None  # Will be set once found

for root in candidate_roots:
    cand_images = os.path.join(root, "images")  # Candidate images folder
    cand_masks = os.path.join(root, "masks")  # Candidate masks folder
    if os.path.isdir(cand_images) and os.path.isdir(cand_masks):
        images_dir = cand_images  # Save found images dir
        masks_dir = cand_masks  # Save found masks dir
        break  # Stop searching once found



# Create dataset and dataloader
dataset = SUIMDataset(images_dir=images_dir, masks_dir=masks_dir, transform=image_transforms, target_transform=mask_transforms)  # Build dataset
loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)  # Build dataloader for batching

print(f"Total paired samples: {len(dataset)}")  # Print dataset size
images_batch, masks_batch = next(iter(loader))  # Fetch one batch to verify shapes
print(f"Batch images shape: {images_batch.shape}, Batch masks shape: {masks_batch.shape}")  # Print batch shapes


## Visualization helpers

def denormalize(img):
    # Reverse ImageNet normalization for visualization
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert from CHW to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip to valid range
    return img  # Return displayable image


## Disblay a few random samples (image + mask)

num_to_show = 3  # Number of samples to visualize
indices = random.sample(range(len(dataset)), k=min(num_to_show, len(dataset)))  # Random indices without replacement

for idx in indices:
    img, mask = dataset[idx]  # Load one sample

    # Prepare mask for display: mask is [1,H,W] after PILToTensor; show as [H,W]
    mask_2d = mask.squeeze(0).numpy()  # Convert to 2D numpy array

    # Plot image and mask side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # Create subplot
    axes[0].imshow(denormalize(img))  # Show denormalized image
    axes[0].set_title("Underwater Image")  # Title
    axes[0].axis("off")  # Hide axes

    axes[1].imshow(mask_2d, cmap="gray")  # Show mask (IDs) in grayscale
    axes[1].set_title("Segmentation Mask (Remapped IDs)")  # Title
    axes[1].axis("off")  # Hide axes

    plt.show()  # Render plot


In [ ]:
# TO DO

!pip install -q segmentation_models_pytorch  # Install the required lbrary

import torch  # Import torch to create the device and move the model
import segmentation_models_pytorch as smp  # Import segmentation_models_pytorch to build a pretrained U-Net

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the U-Net model with a pretrained EfficientNet-B1 encoder
model = smp.Unet(
    encoder_name="efficientnet-b1",      # Use EfficientNet-B1 as the encoder backbone
    encoder_weights="imagenet",          # Load ImageNet pretrained weights for the encoder
    in_channels=3,                       # Input images are RGB (3 channels)
    classes=8,                           # SUIM has 8 classes (0..7)
).to(device)                             # Move model to the selected device

print(model)


In [ ]:
# TO DO

import torch  # Import torch for tensor operations and device handling
import torch.nn as nn  # Import nn for loss functions
import torch.optim as optim  # Import optim for optimizers
from tqdm import tqdm  # Import tqdm for progress bars


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    # Put the model in training mode (enables dropout/batchnorm updates)
    model.train()

    total_loss = 0  # Accumulate batch losses to compute epoch average

    # Iterate over the dataloader with a progress bar
    for images, masks in tqdm(dataloader):
        # Move images to device (GPU/CPU)
        images = images.to(device)

        # Move masks to device, remove channel dim, and ensure integer class labels
        # masks comes from PILToTensor -> shape [B,1,H,W], so we squeeze dim=1 -> [B,H,W]
        masks = masks.to(device).squeeze(dim=1).to(torch.long)

        # Forward pass: model outputs logits with shape [B, C, H, W] where C=8
        outputs = model(images)

        # Compute loss (CrossEntropyLoss expects logits [B,C,H,W] and targets [B,H,W])
        loss = criterion(outputs, masks)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagate gradients
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Add current batch loss
        total_loss += loss.item()

    # Return average loss over all batches
    return total_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    # Put the model in evaluation mode (disables dropout)
    model.eval()

    total_loss = 0  # Accumulate batch losses to compute epoch average

    # Disable gradient computation for validation to save memory and speed up
    with torch.no_grad():
        # Iterate over validation dataloader
        for images, masks in dataloader:
            # Move images to device
            images = images.to(device)

            # Move masks to device, remove channel dim, and ensure integer class labels
            masks = masks.to(device).squeeze(dim=1).to(torch.long)

            # Forward pass
            outputs = model(images)

            # Compute validation loss
            loss = criterion(outputs, masks)

            # Accumulate loss
            total_loss += loss.item()

    # Return average validation loss over all batches
    return total_loss / len(dataloader)

In [ ]:



# Define loss function
criterion = nn.CrossEntropyLoss()  # Multi-class segmentation loss

# Define optimizer
optimizer = optim.AdamW(model.parameters(), lr=0.0001)  # AdamW with small LR

num_epochs = 10  # Train for 10 epochs
train_losses = []  # Store train losses
val_losses = []  # Store val losses

for epoch in range(num_epochs):
    # Train for one epoch
    train_loss = train_one_epoch(model, loader, criterion, optimizer, device)

    # Validate after epoch
    val_loss = validate(model, loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Print epoch summary
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

# Plot loss curves
plt.plot(range(1, num_epochs + 1), train_losses, label="Train Loss", marker='o')  # Train curve
plt.plot(range(1, num_epochs + 1), val_losses, label="Validation Loss", marker='o')  # Val curve
plt.xlabel("Epochs")  # X label
plt.ylabel("Loss")  # Y label
plt.title("Training and Validation Loss")  # Title
plt.legend()  # Legend
plt.show()  # Render plot



In [ ]:

# Put model in eval mode for inference
model.eval()

# Choose a few random samples from the validation split for visualization
num_vis = 5  # Number of samples to visualize
val_indices = random.sample(range(len(val_dataset)), k=min(num_vis, len(val_dataset)))  # Random indices

for i in val_indices:
    # Get a sample from the validation subset
    img, mask = val_dataset[i]  # img: [3,H,W], mask: [1,H,W] (remapped)

    # Run model prediction
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))  # Add batch dim -> [1,8,H,W]
        pred_mask = torch.argmax(torch.softmax(logits, dim=1), dim=1).cpu().squeeze(0)  # [H,W] predicted IDs

    # Prepare ground truth mask for display
    gt_mask = mask.squeeze(0)  # [H,W]

    # Plot original image, ground truth, and prediction
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # 3 panels

    axes[0].imshow(denormalize(img))  # Show denormalized image
    axes[0].set_title("Original Image")  # Title
    axes[0].axis("off")  # Hide axes

    axes[1].imshow(gt_mask, cmap="gray")  # Show GT mask
    axes[1].set_title("Ground Truth Mask")  # Title
    axes[1].axis("off")  # Hide axes

    axes[2].imshow(pred_mask, cmap="gray")  # Show predicted mask
    axes[2].set_title("Predicted Mask")  # Title
    axes[2].axis("off")  # Hide axes

    plt.show()  # Render